### Data Generator for nexmark queries

In [ ]:
from pyflink.table import EnvironmentSettings, TableEnvironment
import os

from pathlib import Path

from pyflink.java_gateway import get_gateway
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment, ExplainDetail

gateway = get_gateway()
string_class = gateway.jvm.String
string_array = gateway.new_array(string_class, 0)
stream_env = gateway.jvm.org.apache.flink.streaming.api.environment.StreamExecutionEnvironment
j_stream_exection_environment = stream_env.createRemoteEnvironment(
    "localhost", 
    8082, 
    string_array
)

env = StreamExecutionEnvironment(j_stream_exection_environment)
env.set_parallelism(4)
table_env = StreamTableEnvironment.create(env)

kafka_jar_path = Path("./flink-sql-connector-kafka-4.0.0-2.0.jar").resolve()
nexmark_jar_path = Path("./nexmark-flink-0.3-SNAPSHOT.jar").resolve()

combined_jars = f"{kafka_jar_path.as_uri()};{nexmark_jar_path.as_uri()}"
table_env.get_config().set("pipeline.jars", combined_jars)

generator_ddl = """
CREATE TABLE nexmark_generator (
    event_type INT,
    person ROW<
        id BIGINT, name VARCHAR, emailAddress VARCHAR, creditCard VARCHAR,
        city VARCHAR, state VARCHAR, `dateTime` TIMESTAMP(3), extra VARCHAR>,
    auction ROW<
        id BIGINT, itemName VARCHAR, description VARCHAR, initialBid BIGINT,
        reserve BIGINT, `dateTime` TIMESTAMP(3), expires TIMESTAMP(3),
        seller BIGINT, category BIGINT, extra VARCHAR>,
    bid ROW<
        auction BIGINT, bidder BIGINT, price BIGINT, channel VARCHAR,
        url VARCHAR, `dateTime` TIMESTAMP(3), extra VARCHAR>
) WITH (
    'connector' = 'nexmark',
    'first-event.rate' = '10000',
    'next-event.rate' = '10000',
    'events.num' = '100000000',
    'person.proportion' = '30',
    'auction.proportion' = '60'
);
"""
table_env.execute_sql(generator_ddl)

kafka_sink_ddl = """
CREATE TABLE kafka_sink (
    event_type INT,
    person ROW<
        id BIGINT, name VARCHAR, emailAddress VARCHAR, creditCard VARCHAR,
        city VARCHAR, state VARCHAR, `dateTime` TIMESTAMP(3), extra VARCHAR>,
    auction ROW<
        id BIGINT, itemName VARCHAR, description VARCHAR, initialBid BIGINT,
        reserve BIGINT, `dateTime` TIMESTAMP(3), expires TIMESTAMP(3),
        seller BIGINT, category BIGINT, extra VARCHAR>,
    bid ROW<
        auction BIGINT, bidder BIGINT, price BIGINT, channel VARCHAR,
        url VARCHAR, `dateTime` TIMESTAMP(3), extra VARCHAR>
) WITH (
    'connector' = 'kafka',
    'topic' = 'event-demo',
    'properties.bootstrap.servers' = 'kafka-service.kafka.svc.cluster.local:9092',
    'format' = 'json'
);
"""
table_env.execute_sql(kafka_sink_ddl)

generation_query = """
INSERT INTO kafka_sink
SELECT * FROM nexmark_generator;
"""

table_env.execute_sql(generation_query)